# Using Trackio with Tune

<a id="try-anyscale-quickstart-tune-wandb" href="https://console.anyscale.com/register/ha?render_flow=ray&utm_source=ray_docs&utm_medium=docs&utm_campaign=tune-wandb">
    <img src="../../_static/img/run-on-anyscale.svg" alt="try-anyscale-quickstart">
</a>
<br></br>

(tune-trackio-ref)=

[Trackio](https://huggingface.co/docs/trackio/index) is a open-source, lightweight, free experiment tracking 
Python library built on top of Hugging Face Datasets and Spaces 🤗. It has a local first design and the experiments can be viewed with a Gradio dashboard locally or on the HF Hub

```{image} /images/trackio_logo_full.png
:align: center
:alt: Trackio
:height: 80px
:target: https://huggingface.co/docs/trackio/index/
```

Ray Tune currently offers two lightweight integrations for Weights & Biases.
One is the {ref}`TrackioLoggerCallback <air-trackio-logger>`, which automatically logs
metrics reported to Tune to the Wandb API.

The other one is the {ref}`setup_trackio() <air-trackio-setup>` function, which can be
used with the function API. It automatically
initializes the Wandb API with Tune's training information. You can just use the
Wandb API like you would normally do, e.g. using `wandb.log()` to log your training
process.

```{contents}
:backlinks: none
:local: true
```

## Running A Trackio Example

In the following example we're going to use both of the above methods, namely the `TrackioLoggerCallback` and
the `setup_trackio` function to log metrics.

As the very first step, make sure you're logged in into the HF Hub on all machines you're running your training on:

    hf auth login

We can then start with a few crucial imports:

In [ ]:
import random
import time

import numpy as np
import trackio

import ray
from ray import tune
from ray.air.integrations.trackio import (
    TrackioLoggerCallback,
    setup_trackio,
)
from ray.train import RunConfig, ScalingConfig
from ray.train.torch import TorchTrainer

PROJECT_NAME = "trackio-ray-example"

HF_DATASET_ID = "hfusername/your_dataset_name"
HF_SPACE_ID = "hfusername/your_space_name"

NUM_STEPS = 15

: 

## Tracking Ray Tune Experiments with Trackio

This example demonstrates how to integrate Trackio with Ray Tune using `TrackioLoggerCallback`. The callback automatically captures:

- Metrics reported via `tune.report`
- GPU utilization and system telemetry
- Trial configuration and metadata

All trials are logged under a single project, enabling structured comparison across hyperparameter configurations. Results can be persisted to a dataset or space for analysis and sharing with the community.

In [ ]:
def tune_trainable(config):

    for step in range(NUM_STEPS):

        loss = (config["lr"] * 10) / (step + 1) + random.random()
        accuracy = 1 / (loss + 1e-3)

        # Example artifact
        image = np.random.rand(64, 64, 3)

        tune.report(
            {
                "loss": loss,
                "accuracy": accuracy,
                "image_mean": float(image.mean()),
                "step": step,
            }
        )

        time.sleep(0.2)

In [ ]:
def run_tune_example():

    tuner = tune.Tuner(
        tune_trainable,
        param_space={
            "lr": tune.grid_search([0.001, 0.01, 0.1]),
        },
        run_config=tune.RunConfig(
            name="trackio-ray-tune-demo",
            callbacks=[
                TrackioLoggerCallback(
                    project=PROJECT_NAME,
                    auto_log_gpu=True,
                    gpu_log_interval=5,
                    dataset_id=HF_DATASET_ID,
                    space_id=HF_SPACE_ID,
                )
            ],
        ),
    )

    results = tuner.fit()

    print("\nTune finished\n")
    print(results)

## Applying Ray Train with Trackio

This example demonstrates how to integrate Trackio with Ray Train using setup_trackio, enabling manual logging within a training loop.

Here, we simulate the training loop which logs the scalar metrics such as loss and throughput
Step-wise progress for time-series tracking



In [ ]:
def train_loop(config):

    run = setup_trackio(
        config=config,
        project=PROJECT_NAME,
        auto_log_gpu=True,
        gpu_log_interval=5,
        dataset_id=HF_DATASET_ID,
        space_id=HF_SPACE_ID,
    )

    for step in range(NUM_STEPS):

        loss = 5 / (step + 1) + random.random()
        throughput = random.uniform(50, 150)

        if run:
            run.log(
                {
                    "loss": loss,
                    "throughput": throughput,
                    "step": step,
                },
                step=step,
            )

        sample_image = np.random.rand(64, 64, 3)

        if run:
            run.log(
                {
                    "image_mean": float(sample_image.mean()),
                    "image_std": float(sample_image.std()),
                }
            )

        time.sleep(0.2)

    if run:
        run.finish()

In [ ]:
def run_train_example():

    trainer = TorchTrainer(
        train_loop_per_worker=train_loop,
        train_loop_config={"lr": 0.01},
        scaling_config=ScalingConfig(num_workers=1),
        run_config=RunConfig(name="trackio-ray-train-demo"),
    )

    trainer.fit()

## Execute the example



In [ ]:
ray.init()

print("\nRunning Ray Tune experiment\n")
run_tune_example()

print("\nRunning Ray Train experiment\n")
run_train_example()

print("\nOpening dashboard\n")
print("Run manually if needed:")
print('trackio show --project "trackio-ray-demo"')

trackio.finish()
ray.shutdown()

print("\nExecution completed\n")